# 1.介绍

assignment1 要求手动实现：
1. BPE
2. LLM（gpt 的decoder-only）
3. cross-entropy loss & AdamW optimizer
4. train

# 2.实现

先说一下实现思路，一开始我比较懵逼，完全不知道什么意思，其实就是 adapters.py 把所有需要实现的接口都留出来了，这里以 softmax 为例：
```python
def run_softmax(in_features: Float[Tensor, " ..."], dim: int) -> Float[Tensor, " ..."]:
    """
    Given a tensor of inputs, return the output of softmaxing the given `dim`
    of the input.

    Args:
        in_features (Float[Tensor, "..."]): Input features to softmax. Shape is arbitrary.
        dim (int): Dimension of the `in_features` to apply softmax to.

    Returns:
        Float[Tensor, "..."]: Tensor of with the same shape as `in_features` with the output of
        softmax normalizing the specified `dim`.
    """
    raise NotImplementedError
```
介绍了传入的参数是什么样，你应该实现什么。

In [2]:
from typing import Iterable
import torch
from torch import Tensor
import torch.nn.functional as F  # 只用来对齐参考时自测，不在最终实现中依赖

## 2.1 工程逻辑

其实把所有的实现都写到 adapters.py 中即可，但是既然都学 Language Modeling from Scratch 了，那就按一个完整的工程来进行工作。

下面是我设计的工程目录：
```
tests/
├─ adapters.py                # 仅定义 run_*，内部调用 src/* 的实现
└─ src/
   ├─ nn_utils.py             # softmax / cross_entropy / gradient_clipping
   ├─ optim_sched.py          # AdamW 类选择 / 余弦+warmup LR
   ├─ data.py                 # get_batch
   ├─ io.py                   # save/load checkpoint
   ├─ bpe/
   │   ├─ tokenizer.py        # get_tokenizer
   │   └─ train_bpe.py        # run_train_bpe
   └─ model/
       ├─ attention.py        # sdpa / mha / rope / mha_with_rope
       ├─ layers.py           # linear / embedding / swiglu / rmsnorm
       └─ transformer.py      # block / lm
```


## 2.2 nn_utils

这一部分就是实现 torch 中的各种工具，其中包括：
1. Softmax
2. cross_entropy
3. gradient_clipping

### 2.2.1 Softmax

Softmax：多分类问题的输出层，将一组任意实数转换为一个概率分布


数学定义：假设我们有一个包含K个实数的向量 **z** = ($z_1$, $z_2$, ..., $z_K$)，Softmax函数会计算一个新的向量 **σ(z)** = ($\sigma_1$, $\sigma_2$, ..., $\sigma_K$)，其中每个元素 $\sigma_i$ 的计算公式如下：

$$\sigma(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}} \quad \text{for } i = 1, \dots, K$$

在实际 PyTorch 实现中 Softmax 实现有多种优化。这里我们只说数值稳定性优化：“减去最大值技巧 (Subtract-Max Trick)”，这是Softmax实现中**最重要**的优化，用于防止计算过程中的数值上溢（overflow）和下溢（underflow）。

Softmax的核心计算是 $e^{z\_i}$。如果输入的logits向量 `z` 中包含较大的数值（例如，`z_i = 1000`），$e^{1000}$ 的结果会是一个巨大的数字，超出浮点数能表示的范围，导致**上溢 (Overflow)**，结果变为 `inf`。这会导致最终的概率分布变成 `[nan, nan, ...]`。

反之，如果logits都为非常小的负数（例如，`z_i = -1000`），$e^{-1000}$ 的结果会无限接近于0，导致**下溢 (Underflow)**。如果分子和分母都下溢为0，最终结果也会是 `nan`。

PyTorch 给输入向量的所有元素加上或减去同一个常数，其输出结果保持不变。

$$\sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}} = \frac{C \cdot e^{z_i}}{C \cdot \sum_{j=1}^{K} e^{z_j}} = \frac{e^{z_i + \log(C)}}{\sum_{j=1}^{K} e^{z_j + \log(C)}}$$

我们可以选择一个特定的常数 `C` 来优化计算。最佳选择是令 $\\log(C) = - \\max(\\mathbf{z})$，即从所有logits中减去它们的最大值。

$$\sigma(\mathbf{z})_i = \frac{e^{z_i - \max(\mathbf{z})}}{\sum_{j=1}^{K} e^{z_j - \max(\mathbf{z})}}$$

**这样做的好处是：**

1.  **防止上溢**：变换后的向量中，最大的元素是 `0` ($e^0=1$)，所有其他元素都是负数。这样就保证了指数计算的最大结果是 `1`，有效避免了上溢。
2.  **减少下溢风险**：通过将整个数值范围向上“平移”，至少保证了有一个元素（即原始最大值对应的元素）的指数结果为 `1`，使得分母至少为 `1`，从而避免了分母因所有项都下溢成零而导致的除零错误。


In [3]:
def softmax(in_features: Tensor, dim: int) -> Tensor:
    shifted = in_features - infeatures.max(dim=dim, keepdim=True).values
    exps = torch.exp(shifted)
    return exps / exps.sum(dim=dim, keepdim=True)

### 2.2.2 Cross_entropy

在信息论中，**熵 (Entropy)** 用来衡量一个概率分布的“不确定性”或“信息量”。一个系统越混乱、越不可预测，它的熵就越高。

  * **直观例子**：
      * **低熵**：一枚“作弊”的硬币，99%的概率是正面。结果非常确定，所以熵很低。
      * **高熵**：一枚均匀的硬币，正反面概率各50%。结果最不确定，所以熵最高。

而**交叉熵 (Cross-Entropy)** 则更进一步，它用来衡量**两个概率分布之间的差异**。具体来说，它衡量的是，当我们**使用一个“错误的”或“近似的”概率分布 `q` 来表示一个“真实的”概率分布 `p` 时，所需要付出的额外信息量（或编码长度）**。

如果近似分布 `q` 与真实分布 `p` 非常接近，那么交叉熵的值就很低。反之，如果 `q` 与 `p` 相差甚远，交叉熵的值就会很高。

**数学定义与公式**：

假设我们有两个离散的概率分布，$p$ 和 $q$。

  * $p$: 真实分布 (True Distribution)。一般为数据的真实标签，表示为 one-hot 编码。
  * $q$: 预测分布 (Predicted Distribution)。模型的输出，通常是经过 Softmax 或 Sigmoid 函数处理后的概率。

交叉熵 $H(p, q)$ 的计算公式如下：

$$H(p, q) = - \sum_{i=1}^{K} p(x_i) \log(q(x_i))$$

其中：

  * $K$ 是所有可能事件（类别）的总数。
  * $p(x\_i)$ 是事件 $x\_i$ 在真实分布 $p$ 中的概率。
  * $q(x\_i)$ 是事件 $x\_i$ 在预测分布 $q$ 中的概率。

**注意**：我们在进行计算的时候不用完全照搬公式来计算，因为真实分布 $p$ 是 one-hot 编码的，假设真实类别是 $c$，那么只有 $p(x_c)=1$，而所有其他的 $p(x_i)=0$ (当 $i \neq c$ 时)。这样一来，上面的求和公式就可以大大简化：

$$
\begin{aligned}
H(p, q) &= -\sum_{i=1}^{V} p(x_i) \log(q(x_i)) \\
        &= -(p(x_1)\log(q(x_1)) + \dots + p(x_c)\log(q(x_c)) + \dots + p(x_V)\log(q(x_V))) \\
        &= -(0 \cdot \log(q(x_1)) + \dots + 1 \cdot \log(q(x_c)) + \dots + 0 \cdot \log(q(x_V))) \\
        &= -\log(q(x_c))
\end{aligned}
$$

所以，交叉熵损失函数最终要计算的，就是**模型预测的正确类别所对应的概率的负对数值**。我们的目标就是让这个损失值越小越好，也就是让 $q_c$ (正确类别的概率) 越接近 1 越好。


In [ ]:
def cross_entropy(inputs: Tensor, targets: Tensor) -> Tensor:
    # inputs: (B, V), targets:(B,)
    logsunexp = torch.logsumexp(inputs, dim=1, keepdim=True)
    log_probs = inputs - logsumexp
    gathered = log_probs.gather(1, targets.view(-1, 1)).squeeze(1)
    return -gathered.mean()